# Stripmap Image Formation Comparison: StripmapPFA vs FFBP

**Consolidates**: `stripmap_pfa.ipynb` + `ffbp_stripmap.ipynb`  
**Source**: `grdl/example/image_processing/`

## Learning Objectives

- Form continuous strip SAR images from long stripmap CPHD collections
- Compare two algorithms: **StripmapPFA** (frequency-domain) vs **FFBP** (time-domain)
- Understand computational trade-offs: speed vs geometric accuracy
- Measure timing and quality metrics for algorithm selection

## Theory: Algorithm Comparison

| Algorithm | Domain | Geometry | Complexity | Best For |
|-----------|--------|----------|------------|----------|
| **StripmapPFA** | Frequency | Approximate (polar grid) | $O(N \log N)$ | Fast processing, near-linear tracks |
| **FFBP** | Time | Exact (hierarchical BP) | $O(N \log N)$ w/ higher constant | Arbitrary geometry, wide beams, DEM integration |

### StripmapPFA: Sub-aperture Mosaicking

1. Partition full aperture into overlapping sub-apertures ($M$ pulses each, overlap fraction $\alpha$)
2. Form each sub-aperture with PFA (polar grid)
3. Mosaic sub-images into continuous strip with Hann-weighted blending

**Key parameter**: Sub-aperture size $M$ controls azimuth resolution (smaller $M$ = coarser resolution, faster processing)

### FFBP: Hierarchical Back-Projection

1. Partition aperture into leaf sub-apertures ($\sim$8 pulses each)
2. Form each leaf via direct back-projection (coarse polar image)
3. Merge leaves via binary tree (upsample + coherent sum)
4. Convert final polar image to rectangular grid

**Key parameter**: `n_angular` controls azimuth sampling density (more samples = finer detail, slower processing)

## Data Requirements

- CPHD file from stripmap collection (long aperture, $N > 1000$ pulses)
- BIOMASS, Sentinel-1, or other stripmap sensor

## Decision Guide

**Use StripmapPFA when**:
- Collection geometry is approximately linear
- Speed is critical (large scenes, batch processing)
- Modest resolution is acceptable

**Use FFBP when**:
- Exact geometry is required (arbitrary flight paths, squinted collections)
- Wide beamwidths or significant topography
- DEM-integrated formation is needed (future GRDL feature)
- Processing time is not critical

## Imports & Environment Validation

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

import grdl
from packaging.version import Version

REQUIRED_VERSION = '0.6.1'
current_version = Version(grdl.__version__)
required_version = Version(REQUIRED_VERSION)

if current_version < required_version:
    raise RuntimeError(
        f"GRDL {REQUIRED_VERSION}+ required. Found {grdl.__version__}. "
        f"Update with: pip install --upgrade grdl"
    )

print(f"✓ GRDL {grdl.__version__} (>= {REQUIRED_VERSION})")

In [ ]:
from pathlib import Path
import time
import gc
import numpy as np
import napari

from grdl.IO import open_reader
from grdl.image_processing.sar.image_formation import StripmapPFA, FastBackProjection

## Configuration

In [ ]:
# Input CPHD file
CPHD_PATH = Path("/path/to/your/stripmap_cphd.cphd")

# StripmapPFA parameters
SUBAPERTURE_SIZE = 512  # Pulses per sub-aperture (smaller = coarser azimuth resolution, faster)
OVERLAP_FRACTION = 0.25  # Sub-aperture overlap (0.0 to 0.5, higher = smoother mosaic)
PIXEL_SPACING = 0.5  # meters (output grid resolution)

# FFBP parameters
N_ANGULAR = 2048  # Angular samples (higher = finer azimuth detail, slower)
N_RADIAL = 2048  # Radial samples (range direction)

# Validate file
if not CPHD_PATH.exists():
    raise FileNotFoundError(f"CPHD file not found: {CPHD_PATH}")
if CPHD_PATH.suffix.lower() not in ['.cphd', '.cph']:
    raise ValueError(f"Expected CPHD file, got {CPHD_PATH.suffix}")

print(f"✓ Input file: {CPHD_PATH.name}")
print(f"\nStripmapPFA settings:")
print(f"  Sub-aperture size: {SUBAPERTURE_SIZE} pulses")
print(f"  Overlap fraction: {OVERLAP_FRACTION}")
print(f"  Pixel spacing: {PIXEL_SPACING} m")
print(f"\nFFBP settings:")
print(f"  Angular samples: {N_ANGULAR}")
print(f"  Radial samples: {N_RADIAL}")

## Load CPHD Metadata

Inspect collection parameters to understand aperture extent and sensor configuration.

In [ ]:
# CRITICAL: Use context manager to prevent CPHD file locks
with open_reader(CPHD_PATH) as reader:
    meta = reader.metadata
    
    # Extract collection parameters
    channel = meta.data.channels[0]
    n_vectors = channel.num_vectors
    n_samples = channel.num_samples
    radar_mode = meta.data.radar_mode
    
    print(f"Collection mode: {radar_mode}")
    print(f"Pulses (vectors): {n_vectors}")
    print(f"Samples per pulse: {n_samples}")
    print(f"\nData size: {n_vectors * n_samples * 8 / (1024**2):.1f} MB (complex64)")
    
    # Compute sub-aperture count for StripmapPFA
    step_size = int(SUBAPERTURE_SIZE * (1 - OVERLAP_FRACTION))
    n_subapertures = max(1, (n_vectors - SUBAPERTURE_SIZE) // step_size + 1)
    
    print(f"\nStripmapPFA will form {n_subapertures} sub-apertures")
    print(f"  Step size: {step_size} pulses")
    print(f"  Overlap: {SUBAPERTURE_SIZE - step_size} pulses")

## Algorithm 1: StripmapPFA (Frequency-Domain Mosaicking)

In [ ]:
print("\nStripmapPFA Image Formation")
print("=" * 60)

# CRITICAL: Use context manager
with open_reader(CPHD_PATH) as reader:
    # Initialize StripmapPFA
    stripmap_pfa = StripmapPFA(
        metadata=reader.metadata,
        subaperture_size=SUBAPERTURE_SIZE,
        overlap_fraction=OVERLAP_FRACTION,
        pixel_spacing=PIXEL_SPACING,
    )
    
    print(f"✓ StripmapPFA initialized")
    print(f"  Output grid: {stripmap_pfa.output_shape} px")
    
    # Form image (timed)
    print(f"\nForming image (this may take several minutes)...")
    t0 = time.time()
    
    stripmap_pfa_image = stripmap_pfa.apply(reader)
    
    elapsed_pfa = time.time() - t0

print(f"\n✓ StripmapPFA complete in {elapsed_pfa:.2f}s")
print(f"  Image shape: {stripmap_pfa_image.shape}")
print(f"  Throughput: {n_vectors / elapsed_pfa:.0f} pulses/sec")
print(f"  Memory usage: {stripmap_pfa_image.nbytes / (1024**2):.1f} MB")

## Algorithm 2: FFBP (Hierarchical Back-Projection)

In [ ]:
print("\nFFBP Image Formation")
print("=" * 60)

# CRITICAL: Use context manager
with open_reader(CPHD_PATH) as reader:
    # Initialize FFBP
    ffbp = FastBackProjection(
        metadata=reader.metadata,
        n_angular=N_ANGULAR,
        n_radial=N_RADIAL,
    )
    
    print(f"✓ FFBP initialized")
    print(f"  Polar grid: {N_RADIAL} × {N_ANGULAR} (radial × angular)")
    
    # Form image (timed)
    print(f"\nForming image (this may take longer than StripmapPFA)...")
    t0 = time.time()
    
    ffbp_image = ffbp.apply(reader)
    
    elapsed_ffbp = time.time() - t0

print(f"\n✓ FFBP complete in {elapsed_ffbp:.2f}s")
print(f"  Image shape: {ffbp_image.shape}")
print(f"  Throughput: {n_vectors / elapsed_ffbp:.0f} pulses/sec")
print(f"  Memory usage: {ffbp_image.nbytes / (1024**2):.1f} MB")
print(f"\nSpeedup ratio (StripmapPFA / FFBP): {elapsed_ffbp / elapsed_pfa:.2f}x")

## Performance Comparison

In [ ]:
print("\nAlgorithm Comparison Summary")
print("=" * 80)
print(f"{'Metric':<30} {'StripmapPFA':<25} {'FFBP':<25}")
print("=" * 80)
print(f"{'Formation time (s)':<30} {elapsed_pfa:<25.2f} {elapsed_ffbp:<25.2f}")
print(f"{'Throughput (pulses/s)':<30} {n_vectors/elapsed_pfa:<25.0f} {n_vectors/elapsed_ffbp:<25.0f}")
print(f"{'Output shape':<30} {str(stripmap_pfa_image.shape):<25} {str(ffbp_image.shape):<25}")
print(f"{'Memory (MB)':<30} {stripmap_pfa_image.nbytes/(1024**2):<25.1f} {ffbp_image.nbytes/(1024**2):<25.1f}")
print(f"{'Algorithm type':<30} {'Frequency-domain':<25} {'Time-domain':<25}")
print(f"{'Geometry':<30} {'Approximate (polar)':<25} {'Exact (hierarchical BP)':<25}")
print("=" * 80)

if elapsed_pfa < elapsed_ffbp:
    print(f"\n→ StripmapPFA is {elapsed_ffbp/elapsed_pfa:.1f}x faster for this collection")
else:
    print(f"\n→ FFBP is {elapsed_pfa/elapsed_ffbp:.1f}x faster for this collection (unusual)")

## Visualization — Side-by-Side Comparison

In [ ]:
viewer = napari.Viewer(title="Stripmap Formation Comparison: PFA vs FFBP")

# Convert to magnitude and dB for display
mag_pfa = np.abs(stripmap_pfa_image)
mag_ffbp = np.abs(ffbp_image)

db_pfa = 20 * np.log10(np.clip(mag_pfa, 1e-12, None))
db_ffbp = 20 * np.log10(np.clip(mag_ffbp, 1e-12, None))

# Add both images
viewer.add_image(
    db_pfa,
    name=f"StripmapPFA ({elapsed_pfa:.1f}s)",
    colormap="gray",
    visible=True,
)

viewer.add_image(
    db_ffbp,
    name=f"FFBP ({elapsed_ffbp:.1f}s)",
    colormap="gray",
    visible=False,  # Toggle to compare
)

# Simplify viewer UI
try:
    viewer.window._qt_viewer.dockLayerControls.setVisible(False)
    viewer.window._qt_viewer.dockConsole.setVisible(False)
    viewer.window._qt_viewer.activityDock.setVisible(False)
except AttributeError:
    pass

print("\n✓ Napari viewer opened")
print("\nViewer tips:")
print("  - Toggle layer visibility to compare StripmapPFA vs FFBP")
print("  - Look for geometric differences (especially at scene edges)")
print("  - FFBP should show better fidelity for non-linear flight paths")
print("  - StripmapPFA may show slight mosaicking artifacts in overlap regions")
print("\nExecution paused — close the napari window to continue and free memory...")

napari.run()

print("\n✓ Viewer closed — cleaning up memory...")
del stripmap_pfa_image, ffbp_image, mag_pfa, mag_ffbp, db_pfa, db_ffbp, viewer
del stripmap_pfa, ffbp
gc.collect()
print("✓ Memory cleanup complete")

## Summary

**GRDL patterns demonstrated**:
- ✅ **StripmapPFA**: Frequency-domain sub-aperture mosaicking for fast processing
- ✅ **FastBackProjection (FFBP)**: Hierarchical time-domain back-projection for exact geometry
- ✅ Performance comparison: timing, throughput, memory usage
- ✅ **Context manager usage** (`with open_reader(CPHD_PATH)`) — prevent OS-level file locks on CPHD files

**Algorithm selection guidance**:

| Criterion | Recommendation |
|-----------|---------------|
| **Linear flight path** | StripmapPFA (faster, sufficient accuracy) |
| **Arbitrary geometry** | FFBP (exact projection) |
| **Large scenes** | StripmapPFA (lower constant factor) |
| **DEM integration** | FFBP (future GRDL support) |
| **Real-time processing** | StripmapPFA (speed critical) |
| **Highest fidelity** | FFBP (no geometric approximations) |

**Key trade-offs**:
- **StripmapPFA**: $2$–$5$× faster, approximate geometry (polar grid), mosaicking artifacts
- **FFBP**: Exact geometry, handles wide beams/arbitrary tracks, higher computational cost

**Tuning parameters**:
- **StripmapPFA**: Reduce `subaperture_size` for faster processing (coarser azimuth resolution)
- **FFBP**: Reduce `n_angular` for faster processing (coarser azimuth sampling)

**Next steps**:
- Export formed images to SICD: see `image_processing/cphd_to_sicd.ipynb`
- Orthorectify formed images: see `ortho/ortho_sicd.ipynb`
- Detect targets: see `image_processing/csi_detection_overlay.ipynb`